# O job das partidas

## Task 4: transformar a carga num job, que roda com um comando a cada Copa nova, e carregar 2026

O pedido do 5.1: partidas e resumo consolidados, carregados **sozinhos** a cada Copa nova. As peças estão prontas;
falta juntá-las num **job**, um script que um agendador chama com um comando.

In [ ]:
import sys
sys.path.insert(0, "../../src")

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F

construtor = (
    SparkSession.builder
    .appName("arena-pipeline")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(construtor).getOrCreate()

RAW = "../../data/raw"
OUT = "../../data/out"

spark

## O script do job

1. **ler** as partidas da pasta do comando; 2. **calcular** o resumo; 3. **gravar** as duas tabelas em Delta, por ano,
com overwrite dinâmico.

In [ ]:
%%writefile ../../src/wc_pipeline/pipeline.py
"""Job das partidas da Arena Analytics (construido na aula 5).

Uso, no terminal do Jupyter, dentro de ~/work:
    spark-submit --master "local[*]" --packages io.delta:delta-spark_4.2_2.13:4.4.0 src/wc_pipeline/pipeline.py data/raw
    spark-submit --master "local[*]" --packages io.delta:delta-spark_4.2_2.13:4.4.0 src/wc_pipeline/pipeline.py data/raw_2026
"""
import sys

from pyspark.sql import SparkSession

from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas, partidas_por_selecao, resumo_por_selecao

PASTA_RAW = sys.argv[1]
SAIDA = "data/out/producao"

spark = (
    SparkSession.builder
    .appName("arena-job-partidas")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{PASTA_RAW}/WorldCupMatches.csv")
)
resumo = resumo_por_selecao(partidas_por_selecao(partidas))

for tabela, df in [("partidas", partidas), ("resumo_selecoes", resumo)]:
    (
        df.repartition("ano").write.format("delta").mode("overwrite")
        .option("partitionOverwriteMode", "dynamic")
        .partitionBy("ano")
        .save(f"{SAIDA}/{tabela}")
    )

print("partidas na saída:", spark.read.format("delta").load(f"{SAIDA}/partidas").count())
spark.stop()

Novidades: **`sys.argv[1]`** (a pasta que vem no comando); a sessão **sem `master`** (quem executa decide onde roda); e
as configurações do Delta **sem** a função do pacote (o pacote vem no comando).

## Executando com spark-submit

`spark-submit` envia um script para o Spark executar. No terminal do Jupyter, em `~/work`:

```bash
spark-submit --master "local[*]" --packages io.delta:delta-spark_4.2_2.13:4.4.0 \\
  src/wc_pipeline/pipeline.py data/raw
```

`--packages` traz o pacote do Delta; `data/raw` é o `sys.argv[1]`. A próxima célula já vem pronta: ela só apaga a saída
anterior do job.

In [ ]:
import shutil
shutil.rmtree(f"{OUT}/producao", ignore_errors=True)

Primeiro, o histórico:

In [ ]:
!cd ../.. && PYTHONPATH=src spark-submit --master "local[*]" --packages io.delta:delta-spark_4.2_2.13:4.4.0 src/wc_pipeline/pipeline.py data/raw 2>/dev/null

## A Copa de 2026

O mesmo script, com a pasta de 2026, **duas vezes**:

In [ ]:
!cd ../.. && PYTHONPATH=src spark-submit --master "local[*]" --packages io.delta:delta-spark_4.2_2.13:4.4.0 src/wc_pipeline/pipeline.py data/raw_2026 2>/dev/null
!cd ../.. && PYTHONPATH=src spark-submit --master "local[*]" --packages io.delta:delta-spark_4.2_2.13:4.4.0 src/wc_pipeline/pipeline.py data/raw_2026 2>/dev/null

Compare o total das duas execuções: um job **idempotente** dá o mesmo resultado ao rodar de novo.

## Conferindo a saída

Como no vídeo 2.3, `assert` para conferir, agora lendo a saída do job:

In [ ]:
partidas_prod = spark.read.format("delta").load(f"{OUT}/producao/partidas")
resumo_prod = spark.read.format("delta").load(f"{OUT}/producao/resumo_selecoes")

total_partidas = partidas_prod.count()
total_copas = partidas_prod.select("ano").distinct().count()
gols_partidas = partidas_prod.select(F.sum("total_gols")).first()[0]
gols_selecoes = resumo_prod.select(F.sum("gols_pro")).first()[0]

assert total_partidas == 964 + 104, "o total de partidas não bate com o histórico mais 2026"
assert total_copas == 23, "falta alguma Copa na saída"
assert gols_partidas == gols_selecoes, "os gols das partidas e das seleções não batem"

print(f"saída conferida: {total_copas} Copas, {total_partidas} partidas e {gols_partidas} gols")

## A consulta da aula 3, agora com 2026

In [ ]:
from pathlib import Path

partidas_prod.createOrReplaceTempView("partidas")
consulta = Path("../../sql/gols_por_copa.sql").read_text(encoding="utf-8")
spark.sql(consulta).filter(F.col("ano") >= 2014).show()

2026 está lá. O público médio vazio: o arquivo de 2026 não traz o público, como 2018 e 2022 (vídeo 2.2).

## Demanda da aula 5 atendida

| Etapa | Entrega |
|---|---|
| Task 1 | `overwrite` em Delta Lake |
| Task 2 | saída por Copa, com partition pruning |
| Task 3 | um arquivo por pasta e overwrite dinâmico |
| Task 4 | `pipeline.py` com `spark-submit`, 2026 carregada sem duplicar, saída conferida |

## Onde cada coisa foi parar

| Aula | Onde ficou | Como é usada a partir de agora |
|---|---|---|
| 1 | `src/wc_pipeline/schemas.py` | o job lê cada Copa nova com o mesmo schema |
| 2 | `src/wc_pipeline/transformacoes.py` | o job chama as três funções a cada carga |
| 3 | `sql/gols_por_copa.sql` | analytics roda a consulta sobre as partidas consolidadas |
| 4 | `data/out/passes_*`, em Parquet | estudo pontual, refeito só se pedirem |
| 5 | `src/wc_pipeline/pipeline.py` e `data/out/producao`, em Delta | roda a cada Copa nova |

Próximo passo: a atividade "Subindo o projeto no Git".

## Para consultar depois

- **Na empresa:** um agendador, como o Airflow, roda o mesmo `spark-submit` a cada Copa ou a cada dia, com a pasta do
  lote no comando. Num cluster (`yarn`, `k8s://...`, Databricks ou EMR), só muda o `--master`: o script continua igual.
- **Na nuvem:** o mesmo código grava num bucket (`s3a://`, `gs://`, `abfss://`), trocando só o caminho. Nunca escreva
  chaves de acesso no código: use variáveis de ambiente, o papel (role) da máquina ou um cofre de segredos.
- **Mais parâmetros:** quando o job precisa de vários parâmetros com nome (`--raw`, `--out`), a biblioteca `argparse`
  do Python organiza a leitura do comando.
- **Opções do `spark-submit`:** `--conf chave=valor` muda uma configuração só naquela execução, e `spark-submit --help`
  lista as outras.
- **Conferências maiores:** em projetos grandes, as conferências com `assert` viram uma suíte de testes automatizados,
  rodada antes de cada mudança no código.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()